In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("krishnakamath/movielens-32m-movies-enriched-with-SIDs", split="train")

# Access an example
example = dataset[0]
print(example['title'])
print(example['genres'])
print(example['all_mpnet_base_v2_embedding'][:5]) # Print first 5 dimensions
print(example['semantic_id'])

# You can iterate through the dataset
for movie in dataset.select(range(5)): # Get first 5 movies
    print(f"Title: {movie['title']}, SID: {movie['semantic_id']}")


In [ ]:
df = pd.DataFrame(dataset)
df

In [ ]:
df['words_in_plot_summary'] = df['plot_summary'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10, 6))
sns.histplot(df['words_in_plot_summary'], bins=30, kde=True)
plt.title('Distribution of Words in Plot Summaries')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Expected embedding dimension for 'all-mpnet-base-v2'
EMBEDDING_DIM = 768

# Filter out rows with malformed embeddings (not a list or incorrect length)
# Ensure the item is a list before checking its length
df_filtered = df[df['all_mpnet_base_v2_embedding'].apply(lambda x: isinstance(x, list) and len(x) == EMBEDDING_DIM)].copy()

# Convert embeddings to numpy array from the filtered DataFrame
embeddings = np.array(df_filtered['all_mpnet_base_v2_embedding'].tolist())

# Function to find similar movies
def find_similar_movies(movie_title, n=5):
    """Find n most similar movies based on embedding similarity"""
    # Use the filtered DataFrame for movie title lookup
    if movie_title not in df_filtered['title'].values:
        return f"Movie '{movie_title}' not found"

    movie_idx = df_filtered[df_filtered['title'] == movie_title].index[0]
    movie_embedding = embeddings[df_filtered.index.get_loc(movie_idx)].reshape(1, -1)

    # Calculate similarity with all movies
    similarities = cosine_similarity(movie_embedding, embeddings)[0]

    # Get top n similar (excluding the movie itself)
    similar_indices = np.argsort(similarities)[::-1][1:n+1]

    # Map back to the original (filtered) DataFrame's index for results
    original_indices = df_filtered.iloc[similar_indices].index

    results = df_filtered.loc[original_indices][['title', 'genres']].copy()
    results['similarity'] = similarities[similar_indices]
    return results

# Test it
find_similar_movies('Toy Story (1995)', n=5)

In [ ]:
# Semantic ID Analysis - group movies by semantic cluster
def movies_by_semantic_id(semantic_id):
    """Get all movies with a specific semantic ID"""
    # Convert semantic_id list to tuple for matching
    matching = df[df['semantic_id'].apply(lambda x: semantic_id in x)]
    return matching[['title', 'genres', 'semantic_id']].head(10)

# Example: Find movies in the first semantic cluster
first_sid = df['semantic_id'].iloc[0][0]
print(f"Movies in semantic cluster {first_sid}:")
movies_by_semantic_id(first_sid)


In [ ]:
# Visualize embedding space with dimensionality reduction
from sklearn.manifold import TSNE

# Reduce embeddings to 2D for visualization (sample for speed)
sample_size = 2000
# Correctly generate sample_indices from the filtered DataFrame length
sample_indices = np.random.choice(len(df_filtered), sample_size, replace=False)
sample_embeddings = embeddings[sample_indices]
# Sample from the filtered DataFrame for consistency
sample_df = df_filtered.iloc[sample_indices].copy()

print("Running t-SNE... (this may take a minute)")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(sample_embeddings)

sample_df['tsne_1'] = embeddings_2d[:, 0]
sample_df['tsne_2'] = embeddings_2d[:, 1]

# Plot with genre coloring
plt.figure(figsize=(14, 10))

# Get primary genre for coloring
sample_df['primary_genre'] = sample_df['genres'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown')

genres = sample_df['primary_genre'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(genres)))

for genre, color in zip(genres[:15], colors):  # Plot top 15 genres
    mask = sample_df['primary_genre'] == genre
    plt.scatter(sample_df[mask]['tsne_1'], sample_df[mask]['tsne_2'],
               label=genre, alpha=0.6, s=30, color=color)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Movie Embedding Space (t-SNE Visualization)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

In [ ]:
# Semantic search: Find movies by natural language query
# You'll need to encode queries using the same model (all-mpnet-base-v2)
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer('all-mpnet-base-v2')

def semantic_search(query, n=5):
    """Find movies matching a natural language query"""
    # Encode the query
    query_embedding = model.encode(query, convert_to_numpy=True)

    # Calculate similarity with all movies
    similarities = cosine_similarity([query_embedding], embeddings)[0]

    # Get top n results
    top_indices = np.argsort(similarities)[::-1][:n]

    results = df.iloc[top_indices][['title', 'genres', 'plot_summary']].copy()
    results['relevance_score'] = similarities[top_indices]
    return results

# Try it!
semantic_search("a story about love and death")
